# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ASP-31/flyrank-int/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata
from huggingface_hub import snapshot_download

# 1. Retrieve Hugging Face token in Google Colab
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

# 2. Download the mid-panel month files locally from Hugging Face
local_dir = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns="*2026-03*",  # Downloads matching month files
    token=hf_token
)

print(f"Downloaded files to: {local_dir}")

# 3. Setup DuckDB and load local parquet files (no recursive argument needed)
con = duckdb.connect()

df_month = con.execute(f"""
    SELECT *
    FROM read_parquet('{local_dir}/**/*.parquet')
""").df()

print("Data loaded successfully! Shape:", df_month.shape)
df_month.head()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Downloaded files to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded successfully! Shape: (9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Field Classification:

Feature: impressions, historical_ctr, query_length, is_branded, device

Label: clicks (or downstream metric next_period_clicks)

Context: property_id, date, country, page, query

Excluded: position (excluded to prevent target feedback loop/leakage during preliminary ranking features).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Print schema and confirm field categorization
print("Available Schema:")
print(df_month.dtypes)

Available Schema:
report_date                 datetime64[us]
client_hash_id                      object
content_hash_id                     object
client_has_gsc                        bool
client_has_ga4                        bool
gsc_data_available                    bool
ga4_data_available                 boolean
gsc_impressions                      int64
gsc_clicks                           int64
gsc_sum_position                     int64
gsc_avg_position                   float64
ga4_pageviews                        Int64
ga4_sessions                         Int64
ga4_users                            Int64
ga4_engaged_sessions                 Int64
ga4_total_engagement_sec             Int64
sessions_organic                     Int64
sessions_direct                      Int64
sessions_referral                    Int64
sessions_social                      Int64
sessions_paid                        Int64
sessions_ai                          Int64
ai_chatgpt                          

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Facts:

Grain Check: Confirming (property_id, query, page, country, device, date) forms a unique primary key per row.

Row Count & Date Span: Verifying total rows and date bounds for the 2026-03 slice.

Availability (Filter with IS TRUE): Checking survival rate for complete non-null records.

5-Feature Frame: Features knowable at decision moment (e.g., historical impression counts, query character length, brand keyword flags).

The Trap (Feature Leakage): Adding a label-derived column (clicks / (impressions + 1)), checking score jump, then removing it.

Code Cell 3

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fact 1: Grain Verification (One row = one report_date + client_hash_id + content_hash_id)
grain_check = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT (report_date || client_hash_id || content_hash_id)) as unique_grain_rows
    FROM df_month
""").df()
print("--- Fact 1: Grain Check ---")
print(grain_check)

# Fact 2: Row Count and Date Span
span_check = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as min_date,
        MAX(report_date) as max_date
    FROM df_month
""").df()
print("\n--- Fact 2: Slice Row Count & Date Span ---")
print(span_check)

# Fact 3: Data Availability (Filter with IS TRUE)
availability_check = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(CASE WHEN (gsc_data_available IS TRUE) THEN 1 END) as gsc_surviving_rows,
        COUNT(CASE WHEN (ga4_data_available IS TRUE) THEN 1 END) as ga4_surviving_rows
    FROM df_month
""").df()
print("\n--- Fact 3: Data Availability (IS TRUE) ---")
print(availability_check)

# 5-Feature Frame (Features knowable at decision time)
feature_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,

        -- Feature 1: High impression content flag (knowable from historical volume)
        CASE WHEN gsc_impressions > 100 THEN 1 ELSE 0 END as feat_is_high_volume,

        -- Feature 2: GSC data connected status
        CASE WHEN client_has_gsc IS TRUE THEN 1 ELSE 0 END as feat_has_gsc,

        -- Feature 3: GA4 data connected status
        CASE WHEN client_has_ga4 IS TRUE THEN 1 ELSE 0 END as feat_has_ga4,

        -- Feature 4: Weekend reporting date flag (calendar feature known at request time)
        CASE WHEN EXTRACT(DAYOFWEEK FROM report_date) IN (0, 6) THEN 1 ELSE 0 END as feat_is_weekend,

        -- Feature 5: Day of month (knowable temporal feature)
        EXTRACT(DAY FROM report_date) as feat_day_of_month,

        -- Label
        gsc_clicks
    FROM df_month
    LIMIT 1000
""").df()

print("\n--- 5-Feature Frame Sample ---")
print(feature_frame.head())

# --- THE TRAP: Feature Leakage Experiment ---
# Adding label-derived feature on purpose
feature_frame['leaked_feature'] = feature_frame['gsc_clicks'] * 0.99  # Directly leaks label

from sklearn.metrics import r2_score

# Score with leak
print("\n--- Leakage Experiment ---")
print("R² Score WITH Leaked Feature:", r2_score(feature_frame['gsc_clicks'], feature_frame['leaked_feature']))

# Drop the leaked feature (keeping honest dataset)
feature_frame = feature_frame.drop(columns=['leaked_feature'])
print("Leaked column successfully dropped for honest pipeline.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Fact 1: Grain Check ---
   total_rows  unique_grain_rows
0     9841378            9841378

--- Fact 2: Slice Row Count & Date Span ---
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

--- Fact 3: Data Availability (IS TRUE) ---
   total_rows  gsc_surviving_rows  ga4_surviving_rows
0     9841378             3611061              413966

--- 5-Feature Frame Sample ---
            client_hash_id           content_hash_id report_date  \
0  client_73cda7b4e4f265ea  content_b7e512995f79d5a6  2026-03-01   
1  client_73cda7b4e4f265ea  content_05597932fe4da067  2026-03-01   
2  client_73cda7b4e4f265ea  content_7a105f548d9c6916  2026-03-01   
3  client_73cda7b4e4f265ea  content_905aa32a0230694e  2026-03-01   
4  client_73cda7b4e4f265ea  content_a3ea9792f793ec72  2026-03-01   

   gsc_impressions  feat_is_high_volume  feat_has_gsc  feat_has_ga4  \
0               20                    0             1             0   
1                1                    0             1

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limitation:
Unbalanced History & Anonymized Queries: Search Console heavily truncates low-volume/long-tail queries for user privacy, creating missing query strings for lower impression buckets. Additionally, early historical data only contains Google Search Console records without linked Analytics conversion telemetry.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check proportion of hidden/anonymized long-tail query rows
# Check proportion of rows missing GA4 analytics telemetry
limitation_check = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 END) as rows_without_ga4,
        COUNT(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 END) * 100.0 / COUNT(*) as pct_missing_ga4_telemetry
    FROM df_month
""").df()

print("--- Limitation Query Output ---")
print(limitation_check)

--- Limitation Query Output ---
   total_rows  rows_without_ga4  pct_missing_ga4_telemetry
0     9841378           9427412                  95.793618


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.